[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-04.ipynb)

# 4주차 실습: 프롬프트와 문맥학습 - 프롬프트 자동 채점

**목표.** 작은 한국어 언어모형을 **가중치 갱신 없이** 그대로 써서, 제주 관광 문의 분류 과제를 **프롬프트 세 가지**로 풀어 보고 자동 채점으로 순위를 매긴다. 그다음 **내 프롬프트 하나를 추가해** 순위가 어떻게 바뀌는지 확인한다.

강의 노트 4주차의 프롬프트 A, B, C와 같은 내용이다. 모델이 다음 토큰의 확률만 계산한다는 점에 주목하자.

## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. GPU는 필요 없다.

In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers torch

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 채점셋 - 제주 관광 문의 24건

정답(라벨)이 있는 평가 데이터다. 자동 채점은 시험 채점과 같아서, **정답지가 있어야** 코드가 점수를 매길 수 있다. 라벨은 주차, 시설, 요금 세 가지다.

In [ ]:
# 제주 관광 문의 채점셋 (라벨: 주차 / 시설 / 요금)
eval_set = [
    ("성산일출봉 주차장이 어디예요?", "주차"),
    ("함덕해수욕장에 주차할 곳이 있나요?", "주차"),
    ("제주공항에서 렌터카를 어디서 반납하나요?", "주차"),
    ("만장굴 주차 요금이 있나요?", "주차"),
    ("협재해수욕장 주차장이 만차인지 알 수 있나요?", "주차"),
    ("한라산 어리목 탐방로 주차가 가능한가요?", "주차"),
    ("천지연폭포 주차장에서 입구까지 멀어요?", "주차"),
    ("섭지코지 주차 공간이 넓은가요?", "주차"),
    ("이호테우해수욕장에 샤워실이 있나요?", "시설"),
    ("성산일출봉에 화장실이 많이 있나요?", "시설"),
    ("함덕해수욕장에 파라솔을 빌릴 수 있나요?", "시설"),
    ("제주민속촌에 수유실이 있나요?", "시설"),
    ("한라산 국립공원에 매점이 있나요?", "시설"),
    ("월정리해수욕장에 짐 보관함이 있나요?", "시설"),
    ("만장굴 안을 휠체어가 다닐 수 있나요?", "시설"),
    ("식물원에 유모차 대여가 되나요?", "시설"),
    ("성산일출봉 입장료가 얼마예요?", "요금"),
    ("만장굴 입장료 할인이 있나요?", "요금"),
    ("제주민속촌 가족권 가격이 어떻게 되나요?", "요금"),
    ("식물원 입장권을 온라인으로 사면 더 싼가요?", "요금"),
    ("우도 왕복 배 삯이 얼마인가요?", "요금"),
    ("한라산 트레킹은 무료인가요?", "요금"),
    ("청소년은 입장료가 할인되나요?", "요금"),
    ("오름 이용 요금이 따로 있나요?", "요금"),
]

LABELS = ["주차", "시설", "요금"]
print(f"채점셋: {len(eval_set)}건")
for lab in LABELS:
    print(f"  {lab}: {sum(1 for _, y in eval_set if y == lab)}건")

### 1-2. 모델 로딩 - 가중치는 그대로

작은 한국어 GPT 모델을 쓴다. 3주차와 달리 **학습시키지 않는다.** 난수도, 데이터로의 갱신도 없이, 다운로드한 가중치 그대로다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 파라미터: {total:,}")
print(f"이번 주에 갱신하는 파라미터: 0 (모델은 그대로, 프롬프트만 바꾼다)")

### 1-3. 라벨 확률로 분류하기 - P(다음 토큰 | 문맥)

모델에게 글을 이어 쓰라고 시키는 대신, **프롬프트 바로 다음 토큰 자리의 확률분포**를 열어 각 라벨의 첫 토큰 확률을 비교한다. 강의 노트의 "모델이 '주차' 토큰에 가장 높은 확률을 두면 그것이 답"과 같은 방식이다. 수가 클수록(0에 가까울수록) 모델이 보기에 그 라벨이 더 자연스럽다

In [ ]:
import torch

def label_scores(prompt_text):
    # 프롬프트 바로 다음 토큰 자리에서 각 라벨 첫 토큰의 로그확률을 비교한다
    prefix_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(prefix_ids).logits
    next_token_logps = torch.log_softmax(logits[0, -1], dim=-1)
    return {lab: next_token_logps[tokenizer(" " + lab, add_special_tokens=False).input_ids[0]].item()
            for lab in LABELS}

# 동작 확인: 지시문만 있는 프롬프트로 한 건 맞혀 보기
trial = "아래 제주 관광 문의를 주차, 시설, 요금 중 하나로 분류하시오.\n\n문의: 섭지코지 주차 공간이 넓은가요?\n라벨:"
trial_scores = label_scores(trial)
for lab, sc in trial_scores.items():
    print(f"  {lab}: {sc:.3f}")
print("-> 모델의 선택:", max(trial_scores, key=trial_scores.get))


### 1-4. 프롬프트 A, B, C - 강의 노트와 같은 세 가지

같은 과제다. A는 지시문만(zero-shot), B는 예시 2개(few-shot), C는 예시는 같은데 서식만 다르다.

In [ ]:
PROMPT_A = """아래 제주 관광 문의를 주차, 시설, 요금 중 하나로 분류하시오.
라벨만 답하시오.

문의: {query}
라벨:"""

PROMPT_B = """아래 제주 관광 문의를 주차, 시설, 요금 중 하나로 분류하시오.
라벨만 답하시오.

문의: 함덕해수욕장 주차장이 어디예요?
라벨: 주차

문의: 오름 입장료 할인이 있나요?
라벨: 요금

문의: {query}
라벨:"""

PROMPT_C = """[과제] 제주 관광 문의 분류 (주차/시설/요금)

Q: 함덕해수욕장 주차장이 어디예요?
A: 주차

Q: 오름 입장료 할인이 있나요?
A: 요금

Q: {query}
A:"""

print(PROMPT_B.format(query="성산일출봉 입장료가 얼마예요?"))

# 같은 문의를 A와 B로 맞혀 본다. 예시가 붙은 쪽(B)이 "요금"을 얼마나 더 자연스럽게 보는지 확인
query = "성산일출봉 입장료가 얼마예요?"
for tag, p in [("A (지시문만)", PROMPT_A), ("B (예시 2개)", PROMPT_B)]:
    sc = label_scores(p.format(query=query))
    print(f"[{tag}] " + "  ".join(f"{k} {v:.2f}" for k, v in sc.items()) + f"  -> 선택: {max(sc, key=sc.get)}")

### 1-5. 자동 채점 실행 - 순위표

채점셋 전체를 돌려 프롬프트별 정확도를 매기고 순위를 출력한다. 수백 번의 채점도 코드가 하니 빠르다.

강의 노트의 막대그래프 수치(A 0.62, B 0.83, C 0.58)는 **설명용 예시**라 여기서 나오는 숫자와 다를 수 있다. 볼 것은 절대값이 아니라 **B와 C의 차이**다. 둘은 담은 정보가 같고 서식만 다른데도 점수가 갈린다.

In [ ]:
def evaluate(prompt, name):
    correct = 0
    per_label = {lab: [0, 0] for lab in LABELS}
    for sentence, gold in eval_set:
        scores = label_scores(prompt.format(query=sentence))
        pred = max(scores, key=scores.get)
        per_label[gold][1] += 1
        if pred == gold:
            correct += 1
            per_label[gold][0] += 1
    acc = correct / len(eval_set)
    breakdown = " ".join(f"{lab} {a}/{b}" for lab, (a, b) in per_label.items())
    return {"이름": name, "정답": correct, "정확도": acc, "라벨별": breakdown}

def print_ranking(candidates):
    results = [evaluate(p, name) for name, p in candidates.items()]
    results.sort(key=lambda r: -r["정확도"])
    print(f"{'순위':<4} {'프롬프트':<18} {'정답':<9} 정확도   라벨별 정답")
    for i, r in enumerate(results, 1):
        print(f"{i:<4} {r['이름']:<18} {r['정답']}/{len(eval_set)}{'':<4} {r['정확도']:.2f}    {r['라벨별']}")

CANDIDATES = {
    "A (지시문만)": PROMPT_A,
    "B (예시 2개)": PROMPT_B,
    "C (서식만 다름)": PROMPT_C,
}

print_ranking(CANDIDATES)


### 1-6. (감각 잡기) 실제로 이어 쓰는 모습 보기

확률 비교 말고, 모델이 프롬프트 뒤를 **실제로 이어 쓰게** 해 보자. few-shot 프롬프트 B 뒤에 어떤 토큰이 오는지 본다.

In [ ]:
sample = "월정리해수욕장에 짐 보관함이 있나요?"
ids = tokenizer(PROMPT_B.format(query=sample), return_tensors="pt").input_ids
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=6, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0]))

## 2. 한 지점만 바꿔 보기 - 내 프롬프트 D 추가

아래 셀의 `# TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 규칙: 지시문, 예시, 서식 중 **한 가지만** 바꾼 프롬프트를 만든다. 무엇을 바꿨는지 주석에 적는다. 바꾸기 전 순위표를 적어 두면 비교할 수 있습니다.

In [ ]:
# TODO: None 대신 내 프롬프트 문자열을 넣는다. {query} 자리에 문의가 들어간다.
# 예시 (이 줄은 지우지 말고 참고만 한다):
# prompt_d = PROMPT_B.replace("라벨만 답하시오.", "라벨 한 단어만 답하시오.")
prompt_d = None

# 아래는 그대로 둡니다
if prompt_d is not None:
    CANDIDATES["D (내 프롬프트)"] = prompt_d

print_ranking(CANDIDATES)


## 3. 확인 질문

1. A, B, C 중 무엇이 1등이었나요? 예시가 붙으면(B) 점수가 어떻게 달라졌나요?
2. B와 C는 담긴 정보가 같은데 점수가 갈렸나요? 갈렸다면 무엇 때문에 그렇다고 생각하나요?
3. 내 프롬프트 D를 넣었을 때 순위가 바뀌었나요? D는 A, B, C와 비교해 **무엇 하나만** 달라졌나요?

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 4. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-04/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.